In [ ]:
# | default_exp preprocessing.ocr

In [ ]:
%load_ext autoreload
%autoreload 2

# PDF OCR preprocessing

> Recursively render PDF pages and recognize them with a local Ollama `glm-ocr` model.

Each source PDF produces one UTF-8 Markdown document under a `.md` directory at the source root. Relative directories are preserved, and page comments retain page provenance. The implementation processes one page at a time to avoid competing for local model resources.

In [ ]:
# | export
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from tempfile import NamedTemporaryFile
from typing import Literal

import pymupdf
from ollama import Client
from tqdm.auto import tqdm


In [ ]:
# | export
@dataclass(frozen=True)
class OCRResult:
    """Outcome of attempting to convert one PDF to Markdown."""

    pdf_path: Path
    markdown_path: Path
    status: Literal["processed", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    error: str | None = None

In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"PDF root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"PDF root is not a directory: {root}")
    return root


def _pdf_jobs(root: Path) -> list[tuple[Path, Path]]:
    """Return deterministic source/target pairs and reject target collisions."""
    output_root = root / ".md"
    pdf_files = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() == ".pdf"
        and not path.is_relative_to(output_root)
    ]
    pdf_files.sort(key=lambda path: path.relative_to(root).as_posix().casefold())

    jobs: list[tuple[Path, Path]] = []
    targets: dict[str, Path] = {}
    for pdf_path in pdf_files:
        relative_path = pdf_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative_path
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"PDF output collision: {previous} and {pdf_path} both map to {markdown_path}"
            )
        targets[collision_key] = pdf_path
        jobs.append((pdf_path, markdown_path))
    return jobs

In [ ]:
# | export
def _response_content(response: object) -> str:
    message = getattr(response, "message", None)
    content = getattr(message, "content", None)
    if not isinstance(content, str) or not content.strip():
        raise ValueError("GLM-OCR returned an empty response")
    return content.strip()


def _page_markdown(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: Client,
    model: str,
    prompt: str,
    dpi: int,
) -> str:
    pixmap = page.get_pixmap(dpi=dpi, alpha=False)
    image_bytes = pixmap.tobytes("png")
    response = client.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
                "images": [image_bytes],
            }
        ],
        options={"temperature": 0},
    )
    content = _response_content(response)
    return f"<!-- Page {page_number} -->\n\n{content}"


def _atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_file.write(content)
            temporary_path = Path(temporary_file.name)
        temporary_path.replace(path)
        temporary_path = None
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

In [ ]:
# | export
def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: Client,
    model: str = "glm-ocr",
    prompt: str = "Text Recognition:",
    dpi: int = 200,
    overwrite: bool = False,
) -> OCRResult:
    """Convert one PDF to Markdown with page-level GLM-OCR requests.

    The final Markdown path is replaced only after every page succeeds. Exceptions
    are captured in the returned result so a folder batch can continue.
    """
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    if not model.strip():
        raise ValueError("model must not be empty")
    if not prompt.strip():
        raise ValueError("prompt must not be empty")
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(source, target, "failed", error="Markdown target is not a file")
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")

            page_sections: list[str] = []
            for page_number, page in enumerate(document, start=1):
                page_sections.append(
                    _page_markdown(
                        page,
                        page_number,
                        client=client,
                        model=model,
                        prompt=prompt,
                        dpi=dpi,
                    )
                )
                pages_completed = page_number

        markdown = "\n\n".join(page_sections).rstrip() + "\n"
        _atomic_write_text(target, markdown)
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
def ocr_folder(
    root_folder: Path | str,
    *,
    host: str = "http://127.0.0.1:11434",
    model: str = "glm-ocr",
    dpi: int = 200,
    overwrite: bool = False,
    request_timeout_s: float = 180,
    client: Client | None = None,
) -> list[OCRResult]:
    """Recursively OCR PDFs into a tree rooted at ``root_folder/.md``."""
    root = _resolve_root(root_folder)
    if not host.strip():
        raise ValueError("host must not be empty")
    if not model.strip():
        raise ValueError("model must not be empty")
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")

    jobs = _pdf_jobs(root)
    if not jobs:
        print(f"No PDF files found under {root}")
        return []

    ocr_client = client if client is not None else Client(host=host, timeout=request_timeout_s)
    try:
        ocr_client.show(model)
    except Exception as error:
        raise RuntimeError(
            f"Cannot use Ollama model '{model}' at {host}. "
            f"Ensure Ollama is running and run `ollama pull {model}`. "
            f"Original error: {error}"
        ) from error

    results: list[OCRResult] = []
    for pdf_path, markdown_path in tqdm(jobs, desc="OCR PDFs", unit="pdf"):
        results.append(
            ocr_pdf(
                pdf_path,
                markdown_path,
                client=ocr_client,
                model=model,
                dpi=dpi,
                overwrite=overwrite,
            )
        )

    counts = Counter(result.status for result in results)
    print(
        f"OCR complete: {counts['processed']} processed, "
        f"{counts['skipped']} skipped, {counts['failed']} failed"
    )
    return results

## Configuration and batch execution

Start Ollama and ensure the model is installed with `ollama pull glm-ocr`. Set `PDF_ROOT` to the directory that contains the PDFs, then uncomment the final line to run the batch.

In [ ]:
PDF_ROOT = Path("../res/SN024002")  # Change to the root containing your PDFs.
OLLAMA_HOST = "http://127.0.0.1:11434"
OLLAMA_MODEL = "glm-ocr"
OCR_DPI = 200
REQUEST_TIMEOUT_S = 180
OVERWRITE = False

In [ ]:
# results = ocr_folder(
#     PDF_ROOT,
#     host=OLLAMA_HOST,
#     model=OLLAMA_MODEL,
#     dpi=OCR_DPI,
#     overwrite=OVERWRITE,
#     request_timeout_s=REQUEST_TIMEOUT_S,
# )
# results

## Tests

The tests use temporary PDFs and a fake Ollama client, so they do not require a running service or write to the repository.

In [ ]:
# | hide
from types import SimpleNamespace
from tempfile import TemporaryDirectory

from fastcore.test import test_eq, test_fail


class FakeOllamaClient:
    def __init__(self, responses=(), show_error: Exception | None = None):
        self.responses = list(responses)
        self.show_error = show_error
        self.show_calls = []
        self.chat_calls = []

    def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        return SimpleNamespace(message=SimpleNamespace(content=response))


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page()
        page.insert_text((72, 72), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()

In [ ]:
# | hide
def test_pdf_discovery_and_mapping():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        (root / "nested").mkdir()
        (root / ".md").mkdir()
        make_pdf(root / "B.PDF")
        make_pdf(root / "nested" / "a.pdf")
        make_pdf(root / ".md" / "ignored.pdf")
        (root / "notes.txt").write_text("not a PDF", encoding="utf-8")

        jobs = _pdf_jobs(root)
        test_eq(
            [source.relative_to(root).as_posix() for source, _ in jobs],
            ["B.PDF", "nested/a.pdf"],
        )
        test_eq(
            [target.relative_to(root).as_posix() for _, target in jobs],
            [".md/B.md", ".md/nested/a.md"],
        )


def test_output_collision_is_rejected():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        make_pdf(root / "same.pdf")
        make_pdf(root / "same.PDF")
        test_fail(lambda: _pdf_jobs(root), contains="output collision")


test_pdf_discovery_and_mapping()
test_output_collision_is_rejected()

In [ ]:
# | hide
def test_ocr_pdf_writes_ordered_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "two-pages.pdf"
        markdown_path = root / ".md" / "two-pages.md"
        make_pdf(pdf_path, ("first", "second"))
        client = FakeOllamaClient(("# First", "Second"))

        result = ocr_pdf(pdf_path, markdown_path, client=client)

        test_eq(result.status, "processed")
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            markdown_path.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# First\n\n<!-- Page 2 -->\n\nSecond\n",
        )
        test_eq(len(client.chat_calls), 2)
        for call in client.chat_calls:
            test_eq(call["model"], "glm-ocr")
            test_eq(call["options"], {"temperature": 0})
            test_eq(call["messages"][0]["content"], "Text Recognition:")
            assert isinstance(call["messages"][0]["images"][0], bytes)


def test_skip_and_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "one.pdf"
        markdown_path = root / "one.md"
        make_pdf(pdf_path)
        markdown_path.write_text("existing", encoding="utf-8")

        skipped_client = FakeOllamaClient()
        skipped = ocr_pdf(pdf_path, markdown_path, client=skipped_client)
        test_eq(skipped.status, "skipped")
        test_eq(skipped_client.chat_calls, [])
        test_eq(markdown_path.read_text(encoding="utf-8"), "existing")

        overwritten = ocr_pdf(
            pdf_path,
            markdown_path,
            client=FakeOllamaClient(("replacement",)),
            overwrite=True,
        )
        test_eq(overwritten.status, "processed")
        assert "replacement" in markdown_path.read_text(encoding="utf-8")


test_ocr_pdf_writes_ordered_markdown()
test_skip_and_overwrite()

In [ ]:
# | hide
def test_failures_do_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "empty-response.pdf"
        markdown_path = root / "empty-response.md"
        make_pdf(pdf_path)
        result = ocr_pdf(pdf_path, markdown_path, client=FakeOllamaClient(("  ",)))
        test_eq(result.status, "failed")
        assert "empty response" in (result.error or "")
        assert not markdown_path.exists()
        assert not list(root.glob("*.tmp"))


def test_corrupt_and_encrypted_pdfs_fail_cleanly():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt_pdf = root / "corrupt.pdf"
        corrupt_pdf.write_bytes(b"not a PDF")
        encrypted_pdf = root / "encrypted.pdf"
        make_pdf(encrypted_pdf, password="secret")

        corrupt = ocr_pdf(corrupt_pdf, root / "corrupt.md", client=FakeOllamaClient())
        encrypted = ocr_pdf(encrypted_pdf, root / "encrypted.md", client=FakeOllamaClient())
        test_eq(corrupt.status, "failed")
        test_eq(encrypted.status, "failed")
        assert "password" in (encrypted.error or "").lower()
        assert not (root / "corrupt.md").exists()
        assert not (root / "encrypted.md").exists()


def test_folder_continues_after_a_failed_pdf():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeOllamaClient((RuntimeError("model failure"), "# B"))

        results = ocr_folder(root, client=client)

        test_eq([result.status for result in results], ["failed", "processed"])
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()
        test_eq(client.show_calls, ["glm-ocr"])


test_failures_do_not_publish_markdown()
test_corrupt_and_encrypted_pdfs_fail_cleanly()
test_folder_continues_after_a_failed_pdf()

In [ ]:
# | hide
def test_folder_validation_and_preflight():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        missing = root / "missing"
        test_fail(lambda: ocr_folder(missing), contains="does not exist")

        empty_client = FakeOllamaClient()
        test_eq(ocr_folder(root, client=empty_client), [])
        test_eq(empty_client.show_calls, [])

        make_pdf(root / "document.pdf")
        unavailable_client = FakeOllamaClient(show_error=RuntimeError("offline"))
        test_fail(
            lambda: ocr_folder(root, client=unavailable_client),
            contains="ollama pull glm-ocr",
        )


test_folder_validation_and_preflight()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()